# ESMM: Entire Space Multi-Task Model on Ali-CCP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_dataset_study/blob/main/notebooks/04_aliccp_cvr/02_esmm.ipynb)
[![Paper](https://img.shields.io/badge/Paper-SIGIR%202018-blue)](https://arxiv.org/abs/1804.07931)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the ESMM architecture and its mathematical foundation
2. Implement ESMM from scratch in PyTorch with shared embeddings
3. Implement a naive single-task CVR baseline for comparison
4. Train both models on Ali-CCP data using GPU acceleration
5. Evaluate CTR AUC, CVR AUC, and CTCVR AUC
6. Perform ablation studies on shared embeddings and loss weights

## Prerequisites

- Completed Notebook 01 (data preprocessing)
- Preprocessed data at `data/aliccp/processed/aliccp_processed.pkl`

## Table of Contents

1. [ESMM Theory](#1-esmm-theory)
2. [Setup & Data Loading](#2-setup--data-loading)
3. [PyTorch Dataset & DataLoader](#3-pytorch-dataset--dataloader)
4. [Model Implementation](#4-model-implementation)
5. [Training](#5-training)
6. [Evaluation & Comparison](#6-evaluation--comparison)
7. [Ablation Studies](#7-ablation-studies)
8. [Exercises](#exercises)
9. [Summary & Key Takeaways](#summary--key-takeaways)

In [ ]:
import os
import json
import time
import pickle
import warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, log_loss, roc_curve

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Paths
DATA_DIR = Path('../../data/aliccp')
PROCESSED_DIR = DATA_DIR / 'processed'

# Hyperparameters
EMBEDDING_DIM = 8
HIDDEN_DIMS = [256, 128, 64]
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5
DROPOUT = 0.2

## 1. ESMM Theory

### The Core Insight (Ma et al., Alibaba, SIGIR 2018)

ESMM addresses the **Sample Selection Bias** problem in conversion prediction through an elegant mathematical decomposition:

$$P(\text{conversion} | \text{impression}) = P(\text{click} | \text{impression}) \times P(\text{conversion} | \text{click}, \text{impression})$$

$$\boxed{\text{CTCVR} = \text{CTR} \times \text{CVR}}$$

### Architecture

ESMM has two towers:
1. **CTR Tower**: Predicts $P(\text{click} | \text{impression})$ -- trained on ALL impressions
2. **CVR Tower**: Predicts $P(\text{conversion} | \text{click}, \text{impression})$ -- output multiplied with CTR

Both towers share the same feature embeddings.

### Loss Function

$$\mathcal{L} = \mathcal{L}_{CTR} + \mathcal{L}_{CTCVR}$$

$$\mathcal{L}_{CTR} = -\frac{1}{N}\sum_{i=1}^{N} [y_i^{click} \log(\hat{p}_i^{ctr}) + (1-y_i^{click})\log(1-\hat{p}_i^{ctr})]$$

$$\mathcal{L}_{CTCVR} = -\frac{1}{N}\sum_{i=1}^{N} [y_i^{conv} \log(\hat{p}_i^{ctcvr}) + (1-y_i^{conv})\log(1-\hat{p}_i^{ctcvr})]$$

where $\hat{p}_i^{ctcvr} = \hat{p}_i^{ctr} \times \hat{p}_i^{cvr}$.

> **Concept:** Notice that there is NO direct CVR loss. The CVR tower learns
> entirely through the CTCVR multiplication. This is the key innovation that avoids SSB.

### Why Shared Embeddings Matter

The CTR task has ~4.6% positive rate (abundant signal), while CTCVR has ~0.03% (very sparse).
Shared embeddings let the CVR tower benefit from the rich CTR gradient signal.

## 2. Setup & Data Loading

In [ ]:
# Load preprocessed data from Notebook 01
processed_path = PROCESSED_DIR / 'aliccp_processed.pkl'

if not processed_path.exists():
    raise FileNotFoundError(
        f"Processed data not found at {processed_path.resolve()}. "
        "Please run Notebook 01 first."
    )

with open(processed_path, 'rb') as f:
    data = pickle.load(f)

field_cardinalities = data['field_cardinalities']
all_fields = data['all_fields']
field_dims = [field_cardinalities[f] for f in all_fields]

print(f'Train: {data["X_ids_train"].shape[0]:,} samples')
print(f'Val:   {data["X_ids_val"].shape[0]:,} samples')
print(f'Fields: {len(all_fields)}')
print(f'Field cardinalities: {field_dims}')
print(f'Train CTR: {data["y_click_train"].mean()*100:.2f}%')
print(f'Train CTCVR: {data["y_conv_train"].mean()*100:.4f}%')

## 3. PyTorch Dataset & DataLoader

In [ ]:
# Create tensors and data loaders
X_ids_train = torch.LongTensor(data['X_ids_train'])
X_vals_train = torch.FloatTensor(data['X_vals_train'])
y_click_train = torch.FloatTensor(data['y_click_train'])
y_conv_train = torch.FloatTensor(data['y_conv_train'])

X_ids_val = torch.LongTensor(data['X_ids_val'])
X_vals_val = torch.FloatTensor(data['X_vals_val'])
y_click_val = torch.FloatTensor(data['y_click_val'])
y_conv_val = torch.FloatTensor(data['y_conv_val'])

train_ds = TensorDataset(X_ids_train, X_vals_train, y_click_train, y_conv_train)
val_ds = TensorDataset(X_ids_val, X_vals_val, y_click_val, y_conv_val)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Input dim: {len(field_dims)} fields x {EMBEDDING_DIM} embed = {len(field_dims) * EMBEDDING_DIM}')

## 4. Model Implementation

### 4.1 ESMM Model

> **Pro Tip:** The Ali-CCP features have both feature IDs (categorical) and feature values
> (continuous). We multiply the embedding by the feature value to incorporate both signals.
> This is different from pure categorical embeddings used in simpler datasets.

In [ ]:
class ESMM(nn.Module):
    """Entire Space Multi-Task Model (ESMM).
    
    Key insight: CTCVR = CTR * CVR
    - CTR tower predicts P(click | impression)
    - CVR tower predicts P(conversion | click, impression)
    - CTCVR = CTR * CVR is supervised against conversion on ALL impressions
    
    Both towers share the same feature embeddings.
    
    Args:
        field_dims: List of cardinalities per feature field
        embedding_dim: Embedding dimension per field
        hidden_dims: List of hidden layer sizes for each tower
        dropout: Dropout rate
    """
    
    def __init__(self, field_dims, embedding_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.n_fields = len(field_dims)
        
        # Shared embeddings (one embedding table per field)
        # padding_idx=0: missing features get zero embedding
        self.embeddings = nn.ModuleList([
            nn.Embedding(dim, embedding_dim, padding_idx=0)
            for dim in field_dims
        ])
        
        input_dim = self.n_fields * embedding_dim
        
        # CTR tower: [Linear -> ReLU -> Dropout] x N -> Linear
        ctr_layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            ctr_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        ctr_layers.append(nn.Linear(prev_dim, 1))
        self.ctr_tower = nn.Sequential(*ctr_layers)
        
        # CVR tower: same architecture, separate parameters
        cvr_layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            cvr_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        cvr_layers.append(nn.Linear(prev_dim, 1))
        self.cvr_tower = nn.Sequential(*cvr_layers)
        
        # Weight initialization
        self._init_weights()
    
    def _init_weights(self):
        """Xavier initialization for embeddings and linear layers."""
        for emb in self.embeddings:
            nn.init.xavier_uniform_(emb.weight.data[1:])  # skip padding
        for module in [self.ctr_tower, self.cvr_tower]:
            for m in module:
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    nn.init.zeros_(m.bias)
    
    def forward(self, feat_ids, feat_vals=None):
        """Forward pass.
        
        Args:
            feat_ids: (batch_size, n_fields) LongTensor of feature IDs
            feat_vals: (batch_size, n_fields) FloatTensor of feature values
        
        Returns:
            ctr_pred: P(click | impression)
            cvr_pred: P(conversion | click, impression)
            ctcvr_pred: P(conversion | impression) = ctr * cvr
        """
        # Embed each field and optionally scale by feature value
        emb_list = []
        for i, emb_layer in enumerate(self.embeddings):
            emb = emb_layer(feat_ids[:, i])  # (B, D)
            if feat_vals is not None:
                emb = emb * feat_vals[:, i:i+1]  # scale by value
            emb_list.append(emb)
        
        x = torch.cat(emb_list, dim=1)  # (B, n_fields * D)
        
        # Tower predictions
        ctr_pred = torch.sigmoid(self.ctr_tower(x).squeeze(-1))
        cvr_pred = torch.sigmoid(self.cvr_tower(x).squeeze(-1))
        
        # ESMM multiplication
        ctcvr_pred = ctr_pred * cvr_pred
        
        return ctr_pred, cvr_pred, ctcvr_pred


# Create model
model_esmm = ESMM(field_dims, EMBEDDING_DIM, HIDDEN_DIMS, DROPOUT).to(device)
n_params_esmm = sum(p.numel() for p in model_esmm.parameters())
print(f'ESMM parameters: {n_params_esmm:,}')
print(f'Architecture: {len(field_dims)} fields -> {len(field_dims)*EMBEDDING_DIM}D -> {HIDDEN_DIMS} -> 1')

### 4.2 Naive CVR Baseline (Single-Task)

In [ ]:
class NaiveCVR(nn.Module):
    """Naive single-task conversion prediction model.
    
    This model is trained ONLY on clicked samples to predict conversion.
    It demonstrates the sample selection bias problem:
    - Trains on P(conversion | click) distribution
    - But must predict for all impressions at serving time
    
    Args:
        field_dims: List of cardinalities per feature field
        embedding_dim: Embedding dimension per field
        hidden_dims: List of hidden layer sizes
        dropout: Dropout rate
    """
    
    def __init__(self, field_dims, embedding_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(dim, embedding_dim, padding_idx=0)
            for dim in field_dims
        ])
        
        input_dim = len(field_dims) * embedding_dim
        layers = []
        prev = input_dim
        for dim in hidden_dims:
            layers.extend([nn.Linear(prev, dim), nn.ReLU(), nn.Dropout(dropout)])
            prev = dim
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, feat_ids, feat_vals=None):
        """Predict conversion probability."""
        embs = []
        for i, emb in enumerate(self.embeddings):
            e = emb(feat_ids[:, i])
            if feat_vals is not None:
                e = e * feat_vals[:, i:i+1]
            embs.append(e)
        x = torch.cat(embs, dim=1)
        return torch.sigmoid(self.mlp(x).squeeze(-1))


model_naive = NaiveCVR(field_dims, EMBEDDING_DIM, HIDDEN_DIMS, DROPOUT).to(device)
n_params_naive = sum(p.numel() for p in model_naive.parameters())
print(f'Naive CVR parameters: {n_params_naive:,}')

## 5. Training

> **Pro Tip:** ESMM trains on ALL impressions (not just clicks), which means each epoch
> processes the full dataset. The CTR task serves as regularization for the shared embeddings,
> while the CTCVR task teaches the CVR tower through the multiplication.

In [ ]:
def evaluate_model(model, data_loader, model_type='esmm', device=device):
    """Evaluate model on given data loader.
    
    Returns dict with CTR AUC, CVR AUC (on clicked), CTCVR AUC.
    """
    model.eval()
    
    all_ctr_preds, all_cvr_preds, all_ctcvr_preds = [], [], []
    all_clicks, all_convs = [], []
    
    with torch.no_grad():
        for batch in data_loader:
            ids, vals, clicks, convs = [b.to(device) for b in batch]
            
            if model_type == 'esmm':
                ctr, cvr, ctcvr = model(ids, vals)
                all_ctr_preds.append(ctr.cpu().numpy())
                all_cvr_preds.append(cvr.cpu().numpy())
                all_ctcvr_preds.append(ctcvr.cpu().numpy())
            else:  # naive
                pred = model(ids, vals)
                all_ctcvr_preds.append(pred.cpu().numpy())
            
            all_clicks.append(clicks.cpu().numpy())
            all_convs.append(convs.cpu().numpy())
    
    clicks_np = np.concatenate(all_clicks)
    convs_np = np.concatenate(all_convs)
    ctcvr_preds = np.concatenate(all_ctcvr_preds)
    
    results = {}
    
    # CTCVR AUC (on all impressions)
    results['ctcvr_auc'] = roc_auc_score(convs_np, ctcvr_preds) if convs_np.sum() > 0 else 0.0
    
    if model_type == 'esmm':
        ctr_preds = np.concatenate(all_ctr_preds)
        cvr_preds = np.concatenate(all_cvr_preds)
        
        # CTR AUC
        results['ctr_auc'] = roc_auc_score(clicks_np, ctr_preds)
        
        # CVR AUC (on clicked samples only)
        clicked_mask = clicks_np == 1
        if clicked_mask.sum() > 10 and convs_np[clicked_mask].sum() > 0:
            results['cvr_auc'] = roc_auc_score(
                convs_np[clicked_mask], cvr_preds[clicked_mask]
            )
        else:
            results['cvr_auc'] = 0.0
    else:
        # For naive model, evaluate CVR on clicked subset
        clicked_mask = clicks_np == 1
        if clicked_mask.sum() > 10 and convs_np[clicked_mask].sum() > 0:
            results['cvr_auc'] = roc_auc_score(
                convs_np[clicked_mask], ctcvr_preds[clicked_mask]
            )
        else:
            results['cvr_auc'] = 0.0
    
    return results

In [ ]:
def train_esmm(model, train_loader, val_loader, n_epochs=5, lr=1e-3,
               weight_decay=1e-5, patience=3, device=device):
    """Train ESMM model with early stopping.
    
    Loss = L_CTR + L_CTCVR (both BCE losses on entire impression space)
    """
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = {
        'train_loss': [], 'ctr_loss': [], 'ctcvr_loss': [],
        'ctr_auc': [], 'cvr_auc': [], 'ctcvr_auc': []
    }
    best_ctcvr_auc = 0
    patience_counter = 0
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss, epoch_ctr_loss, epoch_ctcvr_loss = 0, 0, 0
        n_batches = 0
        start = time.time()
        
        for batch in train_loader:
            ids, vals, clicks, convs = [b.to(device) for b in batch]
            optimizer.zero_grad()
            
            ctr_pred, cvr_pred, ctcvr_pred = model(ids, vals)
            
            # CTR loss: supervised on all impressions
            ctr_loss = nn.functional.binary_cross_entropy(ctr_pred, clicks)
            # CTCVR loss: supervised on all impressions
            ctcvr_loss = nn.functional.binary_cross_entropy(ctcvr_pred, convs)
            
            loss = ctr_loss + ctcvr_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_ctr_loss += ctr_loss.item()
            epoch_ctcvr_loss += ctcvr_loss.item()
            n_batches += 1
        
        # Evaluate
        metrics = evaluate_model(model, val_loader, 'esmm', device)
        elapsed = time.time() - start
        
        # Record history
        history['train_loss'].append(epoch_loss / n_batches)
        history['ctr_loss'].append(epoch_ctr_loss / n_batches)
        history['ctcvr_loss'].append(epoch_ctcvr_loss / n_batches)
        history['ctr_auc'].append(metrics['ctr_auc'])
        history['cvr_auc'].append(metrics['cvr_auc'])
        history['ctcvr_auc'].append(metrics['ctcvr_auc'])
        
        # Early stopping
        if metrics['ctcvr_auc'] > best_ctcvr_auc:
            best_ctcvr_auc = metrics['ctcvr_auc']
            patience_counter = 0
            torch.save(model.state_dict(), str(PROCESSED_DIR / 'esmm_best.pt'))
        else:
            patience_counter += 1
        
        print(f'Epoch {epoch+1}/{n_epochs} ({elapsed:.1f}s) | '
              f'Loss: {epoch_loss/n_batches:.4f} (CTR: {epoch_ctr_loss/n_batches:.4f}, '
              f'CTCVR: {epoch_ctcvr_loss/n_batches:.4f}) | '
              f'CTR AUC: {metrics["ctr_auc"]:.4f} | '
              f'CVR AUC: {metrics["cvr_auc"]:.4f} | '
              f'CTCVR AUC: {metrics["ctcvr_auc"]:.4f}')
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    return history

In [ ]:
# Train ESMM
print("=" * 80)
print("TRAINING ESMM")
print("=" * 80)

esmm_history = train_esmm(
    model_esmm, train_loader, val_loader,
    n_epochs=NUM_EPOCHS, lr=LEARNING_RATE, weight_decay=1e-5, patience=3
)

In [ ]:
# Train Naive CVR baseline (on clicked samples only)
print("\n" + "=" * 80)
print("TRAINING NAIVE CVR (Clicked Samples Only)")
print("=" * 80)

# Create clicked-only data loader
click_mask_train = data['y_click_train'] == 1
X_clicked = torch.LongTensor(data['X_ids_train'][click_mask_train])
V_clicked = torch.FloatTensor(data['X_vals_train'][click_mask_train])
y_clicked = torch.FloatTensor(data['y_conv_train'][click_mask_train])

clicked_ds = TensorDataset(X_clicked, V_clicked, y_clicked)
clicked_loader = DataLoader(
    clicked_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

print(f'Clicked training samples: {len(clicked_ds):,}')
print(f'CVR among clicked: {y_clicked.mean()*100:.2f}%')

naive_optimizer = optim.Adam(model_naive.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
naive_history = {'train_loss': []}

for epoch in range(NUM_EPOCHS):
    model_naive.train()
    total_loss = 0
    n_batches = 0
    start = time.time()
    
    for ids, vals, conv in clicked_loader:
        ids, vals, conv = ids.to(device), vals.to(device), conv.to(device)
        naive_optimizer.zero_grad()
        pred = model_naive(ids, vals)
        loss = nn.functional.binary_cross_entropy(pred, conv)
        loss.backward()
        naive_optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    
    elapsed = time.time() - start
    naive_history['train_loss'].append(total_loss / n_batches)
    print(f'  Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.1f}s) | Loss: {total_loss/n_batches:.4f}')

## 6. Evaluation & Comparison

In [ ]:
# Final evaluation
esmm_results = evaluate_model(model_esmm, val_loader, model_type='esmm', device=device)
naive_results = evaluate_model(model_naive, val_loader, model_type='naive', device=device)

print("=" * 70)
print("FINAL EVALUATION RESULTS")
print("=" * 70)
print(f'{"Model":<15} {"CTR AUC":>10} {"CVR AUC":>10} {"CTCVR AUC":>10}')
print('-' * 50)
print(f'{"Naive CVR":<15} {"  -":>10} '
      f'{naive_results["cvr_auc"]:>10.4f} '
      f'{naive_results["ctcvr_auc"]:>10.4f}')
print(f'{"ESMM":<15} '
      f'{esmm_results["ctr_auc"]:>10.4f} '
      f'{esmm_results["cvr_auc"]:>10.4f} '
      f'{esmm_results["ctcvr_auc"]:>10.4f}')

ctcvr_improvement = esmm_results['ctcvr_auc'] - naive_results['ctcvr_auc']
print(f'\nESMM CTCVR improvement over Naive: +{ctcvr_improvement:.4f} AUC')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
ax = axes[0]
ax.plot(esmm_history['train_loss'], 'b-o', label='ESMM Total Loss', markersize=4)
ax.plot(esmm_history['ctr_loss'], 'g--s', label='CTR Loss', markersize=4)
ax.plot(esmm_history['ctcvr_loss'], 'r--^', label='CTCVR Loss', markersize=4)
ax.plot(naive_history['train_loss'], 'k-D', label='Naive CVR Loss', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# AUC curves
ax = axes[1]
ax.plot(esmm_history['ctr_auc'], 'b-o', label='ESMM CTR AUC', markersize=4)
ax.plot(esmm_history['cvr_auc'], 'g-s', label='ESMM CVR AUC', markersize=4)
ax.plot(esmm_history['ctcvr_auc'], 'r-^', label='ESMM CTCVR AUC', markersize=4)
ax.axhline(y=naive_results['ctcvr_auc'], color='black', linestyle='--',
           label=f'Naive CTCVR AUC: {naive_results["ctcvr_auc"]:.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.set_title('Validation AUC', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Final comparison bar chart
ax = axes[2]
models = ['Naive CVR', 'ESMM']
ctcvr_vals = [naive_results['ctcvr_auc'], esmm_results['ctcvr_auc']]
colors = ['#bab0ac', '#4e79a7']
bars = ax.bar(models, ctcvr_vals, color=colors, edgecolor='white', linewidth=2, width=0.5)
for bar, val in zip(bars, ctcvr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('CTCVR AUC')
ax.set_title('CTCVR AUC: Naive vs ESMM', fontsize=13)
ax.set_ylim(0.5, max(ctcvr_vals) + 0.02)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('ESMM Training on Ali-CCP', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Collect predictions for ROC curves
model_esmm.eval()
model_naive.eval()

all_esmm_ctcvr, all_esmm_cvr = [], []
all_naive_pred = []
all_clicks, all_convs_list = [], []

with torch.no_grad():
    for batch in val_loader:
        ids, vals, clicks, convs = [b.to(device) for b in batch]
        ctr, cvr, ctcvr = model_esmm(ids, vals)
        naive_pred = model_naive(ids, vals)
        
        all_esmm_ctcvr.append(ctcvr.cpu().numpy())
        all_esmm_cvr.append(cvr.cpu().numpy())
        all_naive_pred.append(naive_pred.cpu().numpy())
        all_clicks.append(clicks.cpu().numpy())
        all_convs_list.append(convs.cpu().numpy())

esmm_ctcvr_np = np.concatenate(all_esmm_ctcvr)
esmm_cvr_np = np.concatenate(all_esmm_cvr)
naive_pred_np = np.concatenate(all_naive_pred)
clicks_np = np.concatenate(all_clicks)
convs_np = np.concatenate(all_convs_list)

# ROC on all impressions (CTCVR)
ax = axes[0]
fpr, tpr, _ = roc_curve(convs_np, esmm_ctcvr_np)
ax.plot(fpr, tpr, 'b-', linewidth=2,
        label=f'ESMM CTCVR (AUC={esmm_results["ctcvr_auc"]:.4f})')
fpr, tpr, _ = roc_curve(convs_np, naive_pred_np)
ax.plot(fpr, tpr, 'k--', linewidth=2,
        label=f'Naive CVR (AUC={naive_results["ctcvr_auc"]:.4f})')
ax.plot([0, 1], [0, 1], 'gray', linestyle=':', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve: All Impressions', fontsize=13)
ax.legend(loc='lower right')

# ROC on clicked samples (CVR)
ax = axes[1]
clicked = clicks_np == 1
if clicked.sum() > 0 and convs_np[clicked].sum() > 0:
    fpr, tpr, _ = roc_curve(convs_np[clicked], esmm_cvr_np[clicked])
    ax.plot(fpr, tpr, 'b-', linewidth=2,
            label=f'ESMM CVR (AUC={esmm_results["cvr_auc"]:.4f})')
    fpr, tpr, _ = roc_curve(convs_np[clicked], naive_pred_np[clicked])
    ax.plot(fpr, tpr, 'k--', linewidth=2,
            label=f'Naive CVR (AUC={naive_results["cvr_auc"]:.4f})')
ax.plot([0, 1], [0, 1], 'gray', linestyle=':', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve: Clicked Samples Only', fontsize=13)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 7. Ablation Studies

In [ ]:
# Ablation 1: Shared vs Separate Embeddings
print("=" * 70)
print("ABLATION 1: Shared vs Separate Embeddings")
print("=" * 70)

class ESMMSeparateEmbed(nn.Module):
    """ESMM with SEPARATE embeddings for CTR and CVR towers.
    Tests the hypothesis that shared embeddings are beneficial."""
    
    def __init__(self, field_dims, embedding_dim, hidden_dims, dropout=0.2):
        super().__init__()
        n_fields = len(field_dims)
        
        # Separate embedding tables
        self.ctr_embeddings = nn.ModuleList([
            nn.Embedding(d, embedding_dim, padding_idx=0) for d in field_dims
        ])
        self.cvr_embeddings = nn.ModuleList([
            nn.Embedding(d, embedding_dim, padding_idx=0) for d in field_dims
        ])
        
        inp = n_fields * embedding_dim
        def make_tower(inp_dim):
            layers = []
            prev = inp_dim
            for d in hidden_dims:
                layers.extend([nn.Linear(prev, d), nn.ReLU(), nn.Dropout(dropout)])
                prev = d
            layers.append(nn.Linear(prev, 1))
            return nn.Sequential(*layers)
        
        self.ctr_tower = make_tower(inp)
        self.cvr_tower = make_tower(inp)
    
    def forward(self, ids, vals=None):
        ctr_embs = []
        cvr_embs = []
        for i in range(len(self.ctr_embeddings)):
            ce = self.ctr_embeddings[i](ids[:, i])
            ve = self.cvr_embeddings[i](ids[:, i])
            if vals is not None:
                ce = ce * vals[:, i:i+1]
                ve = ve * vals[:, i:i+1]
            ctr_embs.append(ce)
            cvr_embs.append(ve)
        
        ctr = torch.sigmoid(self.ctr_tower(torch.cat(ctr_embs, 1)).squeeze(-1))
        cvr = torch.sigmoid(self.cvr_tower(torch.cat(cvr_embs, 1)).squeeze(-1))
        return ctr, cvr, ctr * cvr


# Train separate-embedding version
torch.manual_seed(SEED)
model_separate = ESMMSeparateEmbed(field_dims, EMBEDDING_DIM, HIDDEN_DIMS, DROPOUT).to(device)
sep_params = sum(p.numel() for p in model_separate.parameters())
print(f'Separate Embeddings params: {sep_params:,} (vs Shared: {n_params_esmm:,})')

sep_history = train_esmm(
    model_separate, train_loader, val_loader,
    n_epochs=NUM_EPOCHS, lr=LEARNING_RATE, weight_decay=1e-5, patience=5
)

sep_results = evaluate_model(model_separate, val_loader, 'esmm', device)
print(f'\nShared Embeddings CTCVR AUC: {esmm_results["ctcvr_auc"]:.4f}')
print(f'Separate Embeddings CTCVR AUC: {sep_results["ctcvr_auc"]:.4f}')
print(f'Difference: {esmm_results["ctcvr_auc"] - sep_results["ctcvr_auc"]:+.4f}')

In [ ]:
# Ablation 2: CTCVR Loss Weight
print("=" * 70)
print("ABLATION 2: CTCVR Loss Weight")
print("=" * 70)

weight_results = {}

for weight in [0.5, 1.0, 2.0, 5.0]:
    print(f'\n--- CTCVR weight = {weight} ---')
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    
    model_w = ESMM(field_dims, EMBEDDING_DIM, HIDDEN_DIMS, DROPOUT).to(device)
    opt_w = optim.Adam(model_w.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    
    for epoch in range(NUM_EPOCHS):
        model_w.train()
        for batch in train_loader:
            ids, vals, clicks, convs = [b.to(device) for b in batch]
            opt_w.zero_grad()
            ctr, cvr, ctcvr = model_w(ids, vals)
            loss = nn.functional.binary_cross_entropy(ctr, clicks) + \
                   weight * nn.functional.binary_cross_entropy(ctcvr, convs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_w.parameters(), 5.0)
            opt_w.step()
    
    w_results = evaluate_model(model_w, val_loader, 'esmm', device)
    weight_results[weight] = w_results
    print(f'  CTR AUC: {w_results["ctr_auc"]:.4f} | '
          f'CVR AUC: {w_results["cvr_auc"]:.4f} | '
          f'CTCVR AUC: {w_results["ctcvr_auc"]:.4f}')

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
weights = list(weight_results.keys())
ctcvr_aucs = [weight_results[w]['ctcvr_auc'] for w in weights]
ctr_aucs = [weight_results[w]['ctr_auc'] for w in weights]

ax.plot(weights, ctcvr_aucs, 'r-o', label='CTCVR AUC', linewidth=2, markersize=8)
ax.plot(weights, ctr_aucs, 'b--s', label='CTR AUC', linewidth=2, markersize=8)
ax.set_xlabel('CTCVR Loss Weight', fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('Effect of CTCVR Loss Weight on Performance', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save model and results
torch.save(model_esmm.state_dict(), str(PROCESSED_DIR / 'esmm_model.pt'))
torch.save(model_naive.state_dict(), str(PROCESSED_DIR / 'naive_cvr_model.pt'))

def convert_for_json(obj):
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

results_to_save = {
    'esmm': {
        'ctr_auc': convert_for_json(esmm_results['ctr_auc']),
        'cvr_auc': convert_for_json(esmm_results['cvr_auc']),
        'ctcvr_auc': convert_for_json(esmm_results['ctcvr_auc']),
        'params': n_params_esmm,
    },
    'naive_cvr': {
        'cvr_auc': convert_for_json(naive_results['cvr_auc']),
        'ctcvr_auc': convert_for_json(naive_results['ctcvr_auc']),
    }
}

with open(PROCESSED_DIR / 'esmm_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)

print('Models and results saved.')
print(f'  ESMM model: {PROCESSED_DIR / "esmm_model.pt"}')
print(f'  Results: {PROCESSED_DIR / "esmm_results.json"}')

---

## Exercises

### Exercise 1: Add Focal Loss for CTCVR
The conversion label is extremely imbalanced (~0.03% positive overall). Implement focal loss
for the CTCVR component and test whether it improves CTCVR AUC.

In [ ]:
# TODO: Exercise 1
# Implement focal loss: FL(p) = -alpha * (1-p)^gamma * log(p)
# Replace the CTCVR BCE loss with focal loss (gamma=2, alpha=0.25)
# Compare AUC with standard BCE

# def focal_loss(pred, target, gamma=2.0, alpha=0.25):
#     bce = nn.functional.binary_cross_entropy(pred, target, reduction='none')
#     p_t = pred * target + (1 - pred) * (1 - target)
#     focal_weight = (1 - p_t) ** gamma
#     return (alpha * focal_weight * bce).mean()

pass

### Exercise 2: Embedding Dimension Study
Try different embedding dimensions (4, 8, 16, 32) and compare:
1. Model size (parameter count)
2. Training speed (seconds per epoch)
3. Final AUC

In [ ]:
# TODO: Exercise 2
# Loop over embed_dims = [4, 8, 16, 32]
# For each, create ESMM model, train for 3 epochs, record metrics

pass

### Exercise 3: What Happens with Direct CVR Loss?
Modify ESMM to add a direct CVR loss (BCE between CVR tower output and conversion label
on clicked samples). Does this help or hurt?

In [ ]:
# TODO: Exercise 3
# Modify the loss to add:
# L_CVR_direct = BCE(cvr_pred[clicked], conv[clicked])
# L_total = L_CTR + L_CTCVR + alpha * L_CVR_direct
# Test with alpha = 0.1, 0.5, 1.0

pass

### Exercise 4: Calibration Analysis
Plot calibration curves for ESMM predictions (predicted vs actual conversion rate in deciles).
Is the ESMM model well-calibrated?

In [ ]:
# TODO: Exercise 4
# Hint: Bin CTCVR predictions into 10 buckets
# For each bucket, compute mean predicted and actual conversion rate
# Plot predicted vs actual; a perfect model lies on the diagonal

pass

---

## Summary & Key Takeaways

### What We Learned

1. **ESMM eliminates Sample Selection Bias**: By using the multiplication trick (CTCVR = CTR x CVR) and training on all impressions, ESMM avoids the distribution mismatch of naive conversion models.

2. **Shared embeddings matter**: Sharing feature embeddings between CTR and CVR towers provides implicit regularization. The CTR task (with ~4.6% positive rate) helps the CVR task learn better representations through shared gradients.

3. **Loss design is critical**: The ESMM loss only contains $\mathcal{L}_{CTR}$ and $\mathcal{L}_{CTCVR}$. Adding a direct CVR loss re-introduces the sample selection bias.

4. **Practical results on Ali-CCP**: ESMM achieves significantly better CTCVR AUC than the naive CVR baseline on the full impression space, which is the real-world evaluation scenario. Reference: CVR AUC ~0.65, CTCVR AUC ~0.62.

5. **Ali-CCP's extreme imbalance**: With only ~0.03% positive CTCVR, even small AUC improvements translate to meaningful business impact at scale.

### Next Steps

In the next notebook, we implement **MMoE** and **PLE** -- advanced multi-task architectures that use expert networks and gating mechanisms for more flexible task routing.